# Développement d'un assistant pour le service de stérilisation de la citadelle

## Import des packages nécéssaires

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pprint

In [3]:
import os
import sys

In [4]:
print(sys.version)

3.13.3 (main, Apr  8 2025, 13:54:08) [Clang 16.0.0 (clang-1600.0.26.6)]


In [5]:
import numpy as np
import pandas as pd

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

## Setup du projet 

In [7]:
from config import LANGSMITH_API_KEY, OPEN_AI_API_KEY

In [8]:
os.environ["LANGSMITH_TRACING"]="true"
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"]=LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"]="assistant-sterilisation "

In [9]:
os.environ["OPENAI_API_KEY"] = OPEN_AI_API_KEY

## Import des documents

Dans cette section, on s'attarde à l'import des documents. Nous avons deux documents PDF. 

- Le premier fait 108 pages
- Le second 229 pages

`PyMuPDFLoader` works better than `PyPDFLoader` for corrupted PDFs because it's built on MuPDF, a C-based rendering engine with superior error handling and PDF repair capabilities. While `PyPDFLoader` is pure Python and widely used, `PyMuPDFLoader` offers better reliability with malformed files, significantly faster performance (3.5-11x faster depending on the task), and more features like built-in OCR and better image/table extraction. For production RAG applications, PyMuPDFLoader is generally the preferred choice due to its robustness and performance, though it does require C dependencies which can complicate installation in some environments. The choice ultimately depends on your specific use case - PyMuPDF for reliability and performance, PyPDF for simpler, pure Python deployments.

In [ ]:
loader1 = PyMuPDFLoader("../documents/fiche_sterilisation.pdf")
loader2 = PyMuPDFLoader("../documents/guide_bonnes_pratiques.pdf")

In [88]:
document1 = loader1.load()
document2 = loader2.load()

In [89]:
document1[0].metadata["total_pages"]

108

In [90]:
document2[0].metadata["total_pages"]

229

In [91]:
#for doc in document1:
#    print(doc)

In [92]:
#for doc in document1:
#    print(f"Document source: {doc.metadata.get('source', 'Unknown')}")
#    print(f"Total pages: {doc.metadata.get('total_pages', doc.metadata.get('page_count', 'Not found'))}")
#    print(f"Starting page: {doc.metadata.get('page', 'Not found')}")
#    print(f"Author: {doc.metadata.get('author', 'Unknown')}")
#    print("---")

In [93]:
#for doc in document2:
#    print(f"Document source: {doc.metadata.get('source', 'Unknown')}")
#    print(f"Total pages: {doc.metadata.get('total_pages', doc.metadata.get('page_count', 'Not found'))}")
#    print(f"Starting page: {doc.metadata.get('page', 'Not found')}")
#    print(f"Author: {doc.metadata.get('author', 'Unknown')}")
#    print("---")



## Chunk des documents

In [94]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [95]:
import tiktoken

In [96]:
test = "Hello, World !"
tokenizer = tiktoken.get_encoding("o200k_harmony")

In [97]:
tokenizer

<Encoding 'o200k_harmony'>

In [98]:
tokenizer.encode(test) # similaire à https://platform.openai.com/tokenizer

[13225, 11, 5922, 1073]

In [99]:
# fonction utilisée pour définir ce qu'on considère être un token et donc définir un split en chunks pertinent
def length_token(text):
    return len(tokenizer.encode(text))

In [100]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 512,
                                          chunk_overlap = 100,
                                          length_function = length_token,
                                          separators=["\n\n", "\n", ". ", " ", ""])

In [101]:
split_document1 = splitter.split_documents(document1)

In [102]:
split_document1[0]

Document(metadata={'producer': 'Acrobat Distiller 6.0 (Windows)', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2009-04-01T16:03:58+02:00', 'source': 'documents/fiche_sterilisation.pdf', 'file_path': 'documents/fiche_sterilisation.pdf', 'total_pages': 108, 'format': 'PDF 1.4', 'title': 'Microsoft Word - fichesterilisation-hygiene_2003-1-2.doc', 'author': 'Acer', 'subject': '', 'keywords': '', 'moddate': '2009-04-01T16:03:58+02:00', 'trapped': '', 'modDate': "D:20090401160358+02'00'", 'creationDate': "D:20090401160358+02'00'", 'page': 0}, page_content='1 \nFICHES DE STÉRILISATION \n \nCes fiches sont parues sous forme d’un numéro thématique de la revue HYGIENES, en 1996 \n« Fiches de stérilisation » \nGOULLET D., DEWEERDT C., VALENCE B., CALOP J. \nHEALTH & CO – BP 14 – 69 144 Rillieux-Crépieux (anciennement SePres Editeur)  \nISSN : 1249-0075 \nElles ont été  mises à jour en 2003 \n \n \nSommaire \n \n \n \nIntroduction \n \nLa stérilisation à l’hôpital. \n \n \nFiche 1 \n \n

In [103]:
split_document2 = splitter.split_documents(document2)

In [104]:
split_document2[0]

Document(metadata={'producer': 'macOS Version 11.0 (assemblage 20A2411) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20211003120623Z00'00'", 'source': 'documents/guide_bonnes_pratiques.pdf', 'file_path': 'documents/guide_bonnes_pratiques.pdf', 'total_pages': 229, 'format': 'PDF 1.4', 'title': 'Guide-bonnes-pratiques-VD', 'author': 'Christophe Lambert', 'subject': '', 'keywords': '', 'moddate': "D:20211003120623Z00'00'", 'trapped': '', 'modDate': "D:20211003120623Z00'00'", 'creationDate': "D:20211003120623Z00'00'", 'page': 0}, page_content='1 \n \nV1 - 2021 \nGuide \nBonnes Pratiques  \nde Stérilisation des  \nDispositifs Médicaux \nRéutilisables')

## Embeddings des documents et stockage dans une base de données vectorielle

Pour la base de données vectorielles en **dev** on va utiliser `Chorma db` pour la **production** on se tournera vers `pgvector`

In [11]:
embedding_model = "text-embedding-3-small" # modèle d'embedding utilisé par les mdoèles modernes

In [12]:
from langchain_openai import OpenAIEmbeddings

In [13]:
embedding = OpenAIEmbeddings(api_key=OPEN_AI_API_KEY, 
                             model=embedding_model)

In [108]:
all_chunks = split_document1 + split_document2

In [109]:
import chromadb

In [114]:
# intialisation de la db
chroma_client = chromadb.PersistentClient("./chroma_db")

In [115]:
# création de la collection
collection = chroma_client.get_or_create_collection("my_db_sterilisation")

In [116]:
texts = [chunk.page_content for chunk in all_chunks]
sources = [{"source": chunk.metadata["source"]} for chunk in all_chunks]


vectors = embedding.embed_documents(texts)

idx = [f"doc{i}" for i in range(len(all_chunks))]
collection.add(
    ids=idx,
    embeddings=vectors,
    documents=texts,
    metadatas=sources,
)

## Retriever

Pour le RAG nous avons besoin de:

- un `retriever`: donc un moyen de récupérer les informations par correspondance sémentique dans la base de données.
- un `generator`: donc un LLM qui va remettre le tout en forme.

Actuellement on ne sait faire que retrouver de l'information de la bdd qui pourrait être passée au LLM sans soucis. Cependant pour des raisons de flexibilité nous allons utiliser LangChain et créer notre `retriever` qui sera passé dans la chaine par la suite.

In [14]:
from langchain_chroma import Chroma

In [15]:
vectore_store = Chroma(collection_name="my_db_sterilisation", 
                       embedding_function=embedding,
                       persist_directory="./chroma_db")

In [16]:
retriever = vectore_store.as_retriever(search_kwargs = {"k":5})

In [17]:
# Création d'un prompt (à partir d'un prompt template)
from langchain_core.prompts import PromptTemplate

In [18]:
prompt_template = """You are a helpful assistant. Answer the question using ONLY the information from the context below. 
Do NOT use any prior knowledge.
If the answer is not in the context, respond exactly with: "I don't have enough information to answer this question."

Context:
{context}

Question: {question}

Answer in French. End your answer with a "Sources:" section. 
For each source, copy EXACTLY the filename and page number from the [Source: filename, page X] tags in the context above.
Example of correct format:
- documents/fiche_sterilisation.pdf, page 4
- documents/autre_document.pdf, page 12"""

In [19]:
prompt = PromptTemplate.from_template(prompt_template)

## Generator

In [20]:
from langchain_openai import ChatOpenAI

In [21]:
from langchain_ollama import OllamaLLM

In [22]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPEN_AI_API_KEY, temperature=0, request_timeout=60)

In [33]:
llm2 = OllamaLLM(model="llama3.1:8b")

## Chaine RAG complète

In [29]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [34]:
# ce qui va sortir du retriver est une liste donc pour passer cela au LLM il faut tout assembler en strings (uniquement la partie information donc page_content)
def format_docs(docs):
    return "\n\n".join(
        f"[Source: {doc.metadata['source']}, page {doc.metadata.get('page', '?')}]\n{doc.page_content}"
        for doc in docs
    )

chain = (
    {"context": retriever|format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm2
    | StrOutputParser()
)

## Tests

In [31]:
question = "Définis en 5 points les principales choses à savoir pour la stérilisation des instruments en milieu hospitalié."

In [35]:
chain.invoke(question)

"Voici les principales choses à savoir pour la stérilisation des instruments en milieu hospitalier :\n\n1. La stérilisation est une spécialisation de l’activité visant à la prévention des infections nosocomiales (Source: documents/fiche_sterilisation.pdf, page 3).\n2. L'obtention de l'état stérile et son maintien jusqu'au moment de l'utilisation constituent une obligation de résultat (Source: documents/guide_bonnes_pratiques.pdf, page 19).\n3. La réutilisation et la restérilisation des dispositifs médicaux à usage unique sont interdites en France (Source: documents/fiche_sterilisation.pdf, page 26).\n4. Les étapes préalables à la stérilisation proprement dite (prétraitement, lavage, recomposition, conditionnement) sont indissociables de cette dernière et doivent conduire à réduire les contaminations microbiologiques, chimiques et particulaires sur le dispositif médical (Source: documents/guide_bonnes_pratiques.pdf, page 19).\n5. Le personnel chargé de la stérilisation doit se protéger 

### Réponse avec gpt-4o-mini

'1. La stérilisation des dispositifs médicaux est essentielle pour prévenir les infections nosocomiales et doit être réalisée dans le cadre d\'un système d\'Assurance Qualité.\n

2. Les opérations de stérilisation comprennent plusieurs étapes : pré-désinfection, nettoyage, conditionnement, stérilisation proprement dite, contrôles, et stockage.\n

3. La stérilisation doit être effectuée dans des locaux adaptés, avec un personnel formé et des procédures de circulation bien établies.\n

4. La stérilité est définie comme l\'absence de tout microorganisme viable, et pour qu\'un dispositif soit étiqueté "stérile", la probabilité de présence d\'un microorganisme viable doit être inférieure ou égale à 1 pour 1 million.\n

5. La réutilisation et la restérilisation de dispositifs médicaux à usage unique sont interdites en France.\n\n

Sources:\n- documents/fiche_sterilisation.pdf, page 4\n- documents/fiche_sterilisation.pdf, page 17\n- documents/fiche_sterilisation.pdf, page 19\n- documents/fiche_sterilisation.pdf, page 26'

### Réponse avec Qwen2.5-coder:7b

"Voici les principales choses à savoir pour la stérilisation des instruments en milieu hospitalié :\n\n

1. Stérilisation et Hygiène Hospitalière sont indissociables.\n

2. La stérilisation est une spécialisation de l'activité visant à prévenir les infections nosocomiales.\n

3. La stérilisation par chaleur sèche est désormais interdite pour la stérilisation des dispositifs médicaux dans les établissements de santé.\n

4. Les opérations de stérilisation des dispositifs médicaux comprennent la pré-désinfection, le nettoyage, la conditionnement, la stérilisation proprement dite et les contrôles des différentes opérations.\n

5. La stérilisation doit conduire à réduire les contaminations microbiologiques, chimiques et particulières sur le dispositif médical traité.\n\n

Sources :\n- documents/fiche_sterilisation.pdf, page 3\n- documents/guide_bonnes_pratiques.pdf, page 19"


### Réponse avec LLama3.1:8b

"Voici les principales choses à savoir pour la stérilisation des instruments en milieu hospitalier :\n\n

1. La stérilisation est une spécialisation de l’activité visant à la prévention des infections nosocomiales (Source: documents/fiche_sterilisation.pdf, page 3).\n

2. L'obtention de l'état stérile et son maintien jusqu'au moment de l'utilisation constituent une obligation de résultat (Source: documents/guide_bonnes_pratiques.pdf, page 19).\n

3. La réutilisation et la restérilisation des dispositifs médicaux à usage unique sont interdites en France (Source: documents/fiche_sterilisation.pdf, page 26).\n

4. Les étapes préalables à la stérilisation proprement dite (prétraitement, lavage, recomposition, conditionnement) sont indissociables de cette dernière et doivent conduire à réduire les contaminations microbiologiques, chimiques et particulaires sur le dispositif médical (Source: documents/guide_bonnes_pratiques.pdf, page 19).\n

5. Le personnel chargé de la stérilisation doit se protéger lors des opérations et respecter les conditions d'utilisation des solutions de stérilisation (Source: documents/fiche_sterilisation.pdf, page 26).\n\n

Sources:\n- documents/fiche_sterilisation.pdf, page 3\n- documents/guide_bonnes_pratiques.pdf, page 19\n- documents/fiche_sterilisation.pdf, page 26"


In [37]:
answer = chain.stream(question)
for chunk in answer:
    print(chunk, end="", flush=True)  # print sans saut de ligne

Voici les principales choses à savoir pour la stérilisation des instruments en milieu hospitalier :

1. La stérilisation est un acte de soins indirect qui vise à prévenir les infections nosocomiales.
2. La stérilisation par chaleur sèche est désormais interdite pour la stérilisation des dispositifs médicaux dans les établissements de santé.
3. Les opérations de stérilisation des dispositifs médicaux comprennent la pré-désinfection, le nettoyage, le conditionnement et la stérilisation proprement dite.
4. La réutilisation et la restérilisation de dispositifs à usage unique sont interdites en France.
5. L'obtention de l'état stérile et son maintien jusqu'au moment de l'utilisation constituent une obligation de résultat.

Sources:
- documents/fiche_sterilisation.pdf, page 3
- documents/fiche_sterilisation.pdf, page 4
- documents/guide_bonnes_pratiques.pdf, page 19
- documents/fiche_sterilisation.pdf, page 26

In [44]:
question2 = "Comment définit-on qu'un instrument est bien stéril  ?"

In [45]:
answer2 = chain.stream(question2)
for chunk in answer2:
    print(chunk, end="", flush=True)  # print sans saut de ligne

Selon la définition donnée dans le document "documents/fiche_sterilisation.pdf, page 7", un instrument est considéré comme bien stérilisé si la probabilité d'occurrence d'un article non stérile dans la population est inférieure ou égale à 1 pour 1 million (10^-6).

Sources:
- documents/fiche_sterilisation.pdf, page 7